In [ ]:
source(here::here("data-cleaning", "00a-parameters.r"))


In [ ]:
# Update the grouper
system("git submodule update --init --recursive")

# List required packages
required_packages <- c(
  "data.table", # Fast data manipulation
  "here", # Simplifies file path management
  "tictoc", # Timing code execution
  "stringr", # String manipulation
  "stringi", # Unicode string processing
  "lubridate", # Date-time handling
  "profvis", # Profiling R code
  "hash", # Hashing utility
  "future", # Parallel processing
  "future.apply", # Parallelized apply functions
  "knitr", # Dynamic report generation
  "htmlwidgets", # Interactive HTML widgets
  "parallelly", # Advanced parallel computing
  "stringdist", # String distance calculations
  "parallel", # Base parallel computing
  "reticulate", # Interface to Python
  "bigrquery", # BigQuery client
  "jsonlite", # JSON parsing
  "googleCloudStorageR", # Google Cloud Storage access
  "haven", # Read/write Stata, SPSS, SAS files
  "fst", # Fast serialization
  "httr", # HTTP requests
  "ggplot2", # Data visualization
  "rmarkdown", # Dynamic markdown documents
  "digest", # Create cryptographic hashes
  "base64enc", # Base64 encoding/decoding
  "arrow", # Apache Arrow for fast data storage
  "tidyverse", # Collection of data science packages,
  "fasttime", # for fastPOSIXct
  "glue", # for string pasting
  "progressr" # live progress and ETA
)

github_packages <- c(
  "r-lib/styler" # Code formatting
)

# Installation commands (commented out, for reference)
invisible(lapply(
  required_packages, function(pkg) {
    if (!require(pkg, character.only = TRUE)) {
      install.packages(pkg)
    }
  }
))
invisible(lapply(
  github_packages, function(repo) {
    if (!require(basename(repo), character.only = TRUE)) {
      remotes::install_github(repo)
    }
  }
))

# Load packages (assumes they are already installed)
invisible(lapply(required_packages, library, character.only = TRUE))
invisible(lapply(basename(github_packages), library, character.only = TRUE))


In [ ]:
year_to_load <- as.numeric(fread("~/pids-drg-claims/data-cleaning/debug/cache/year_to_load.txt"))

# Source each file sequentially
for (file in list.files(
  here::here("data-cleaning/r_scripts_v2"),
  pattern = "\\.R$", full.names = TRUE
)) {
  invisible(source(file))
}

message(year_to_load)


In [ ]:
# Enable caching and printing options for data mapping
to_use_cache <- TRUE # Set to TRUE to enable saving and loading of .rds files
to_print_mapping_data <- FALSE # Set to TRUE to print mapping data tables

# Helper function to load data from cache or query from BigQuery if not cached
load_or_query <- function(
    query, var_name, year_to_load = NULL,
    overwrite_cache = FALSE) {
  rds_path <- here(
    cache_path, "mapping",
    paste0(ifelse(var_name == "hci" & !is.null(year_to_load),
      paste0("hci_", year_to_load),
      ifelse(var_name == "claims" & !is.null(year_to_load),
        paste0("claims_", year_to_load),
        var_name
      )
    ), ".rds")
  )


  if (!overwrite_cache && to_use_cache && file.exists(rds_path)) {
    return(readRDS(rds_path))
  }

  dt <- query_bq_to_dt(query)

  if (to_use_cache) saveRDS(dt, rds_path)
  return(dt)
}

# Helper function to print all rows of a data.table if
# to_print_mapping_data is enabled
if (to_print_mapping_data) {
  print_all <- function(dt, title) {
    cat("\n---", title, "---\n") # Print table title
    print(dt, nrow = Inf) # Print all rows of the data.table
  }
}

# 1. Query and load the tdrg_libraries.proc table
# This table contains procedure codes and attributes
# like description, classification, and site
proc_query <- paste0("SELECT * FROM ", gcp_proj, ".tdrg_libraries.proc")
proc <- load_or_query(proc_query, "proc")
proc[, CODE := as.character(CODE)] # Ensure the CODE column is of character type

# 2. Query and load the phic.acr_rvs_map table
# This table maps RVS codes to ICD-9-CM codes,
# used for healthcare billing purposes
rvs_icd9_query <- paste0(
  "SELECT * FROM ",
  gcp_proj, ".phic_libraries.acr_rvs_map"
)
rvs_icd9 <- load_or_query(rvs_icd9_query, "rvs_icd9")

# Convert RVS and ICD9CM columns to character type
# and adjust ICD9CM for multiplication
rvs_icd9 <- rvs_icd9[, .(
  rvs = as.character(rvs),
  icd9cm = as.character(as.numeric(icd9cm) * 100)
)]

# Merge the RVS-ICD9 mapping with the proc table for DRG classification
rvs_icd9 <- merge(
  rvs_icd9,
  proc[, .(CODE, DRGUSE)], # Select CODE and DRGUSE columns for merging
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
  # Merge on icd9cm and CODE columns
)

# Filter and annotate DRG-related codes, removing unnecessary DRGUSE column
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][
  !is.na(rvs) & !is.na(icd9cm), -"DRGUSE"
]

# 3. Query and load phic.acr_procedure table
# This table contains RVS codes, relative value units (RVUs),
# and descriptions for procedures
acr_rvs_query <- paste0(
  "SELECT * FROM ",
  gcp_proj, ".phic_libraries.acr_procedure"
)
acr_rvs <- load_or_query(acr_rvs_query, "acr_rvs")

# 4. Query and load tdrg_libraries.i10 table
# This table contains ICD-10 codes with DRG grouping data,
# including codes marked as "accepted" (ACCPDX = "Y")
i10_query <- paste0("SELECT * FROM ", gcp_proj, ".tdrg_libraries.i10")
tdrg_icd10 <- load_or_query(i10_query, "tdrg_icd10")
setkey(tdrg_icd10, "CODE") # Set the CODE column as key for efficient lookups

# Extract unique accepted ICD-10 codes for diagnosis
# (ACCPDX == "Y") and store in acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])

# Create an environment for quick lookup of accepted diagnosis codes
acc_pdx_env <- new.env(hash = TRUE, parent = emptyenv())
for (code in acc_pdx) {
  # Assign each accepted code to the environment
  assign(code, TRUE, envir = acc_pdx_env)
}

# 5. Query and load icd.phl_icd10 table
# ICD-10 codes specific to the Philippines
phl_icd10_query <- paste0("SELECT * FROM ", gcp_proj, ".icd.phl_icd10")
phl_icd10 <- load_or_query(phl_icd10_query, "phl_icd10")

# Filter and process neoplasm codes by extracting
# specific codes from complex ICD-10 notations
neoplasms_dt_actual <- as.data.table(phl_icd10[
  # Select rows with '/' in icd10, indicating neoplasm codes
  grepl("/", icd10), .(icd10)
  # Extract relevant part
][, icd10 := sapply(strsplit(icd10, ","), function(x) trimws(x[2]))])


# 6. Query and load tdrg_libraries.i10vx table
# This table contains an expanded version of ICD-10 codes with validation flags
i10vx_query <- paste0("SELECT * FROM ", gcp_proj, ".tdrg_libraries.i10vx")
i10vx <- load_or_query(i10vx_query, "i10vx")
setkey(i10vx, "code") # Set the code column as key for efficient lookup
acc_icd <- unique(i10vx[, code]) # Extract unique ICD codes from this table
acc_icd_set <- unique(acc_icd)
# 7. Query and load hci.temp_hci table
# This table lists healthcare institutions with details
# like ownership, category, and location
# Query and cache HCI data per year_to_load
# hci_query <- paste0("SELECT * FROM ", gcp_proj, paste0(".phic_hci.hci_", year_to_load))
# hci <- load_or_query(hci_query, "hci", year_to_load)
hci_query <- paste0("SELECT * FROM ", gcp_proj, paste0(".phic_hci.hci_full"))
hci <- load_or_query(hci_query, "hci_full")

# 8. Define global variables for use later in the script:
neoplasm_codes <- unique(neoplasms_dt_actual$icd10) # Unique neoplasm codes
covid_codes <- unique(covid_rvs) # Unique COVID-related codes
rvs_codes <- unique(acr_rvs$rvs) # Unique RVS codes

neoplasm_pattern <- paste0("(", paste(neoplasm_codes, collapse = "|"), ")")
covid_pattern <- paste0("(", paste(covid_codes, collapse = "|"), ")")
rvs_pattern <- paste0("(", paste(rvs_codes, collapse = "|"), ")")

phil_icds <- unique(gsub(
  "[^A-Za-z0-9]", "",
  phl_icd10[!grepl("/", icd10), icd10]
))
icd_codes <- unique(tdrg_icd10$CODE)

# Function to create an environment from a vector of unique values
create_env_from_vector <- function(vec) {
  env <- new.env(parent = emptyenv())
  list2env(setNames(as.list(rep(TRUE, length(vec))), vec), envir = env)
  return(env)
}

# 1. Create environment for proc table data if specific values are needed
# Here we assume proc$CODE is the field of interest
proc_env <- create_env_from_vector(proc$CODE)

# 2. Create environment for rvs_icd9 table data based on rvs and icd9cm
rvs_env <- create_env_from_vector(rvs_icd9$rvs)
icd9cm_env <- create_env_from_vector(rvs_icd9$icd9cm)

# 3. Environment for acr_rvs table (assuming rvs is the field of interest)
acr_rvs_env <- create_env_from_vector(acr_rvs$rvs)

# 4. Environment for accepted ICD-10 codes (from tdrg_icd10)
acc_pdx_env <- create_env_from_vector(acc_pdx)

# 5. Environment for phl_icd10 ICD-10 codes (e.g., neoplasm codes)
phl_icd10_env <- create_env_from_vector(phl_icd10$icd10)

# 6. Environment for expanded ICD-10 codes (i10vx)
acc_icd_env <- create_env_from_vector(i10vx$code)

# 7. Environment for hci table data if needed for specific fields (e.g., id_hci)
# Assuming hci$id_hci is the identifier of interest
hci_env <- create_env_from_vector(hci$id_hci)

# 8. Other specific environments for global variables
neoplasm_env <- create_env_from_vector(neoplasm_codes)
covid_env <- create_env_from_vector(covid_codes)
rvs_codes_env <- create_env_from_vector(rvs_codes)
phil_icds_env <- create_env_from_vector(phil_icds)
icd_codes_env <- create_env_from_vector(icd_codes)

# Combine COVID and neoplasm codes into a single environment
covid_neoplasm_codes <- unique(c(covid_codes, neoplasm_codes))
covid_neoplasm_env <- create_env_from_vector(covid_neoplasm_codes)

# Combine COVID, RVS, and neoplasm codes into a
# single environment for efficient lookup
covid_rvs_neoplasm_codes <- unique(c(covid_codes, rvs_codes, neoplasm_codes))
covid_rvs_neoplasm_env <- create_env_from_vector(covid_rvs_neoplasm_codes)

# Create a combined regular expression pattern to match COVID,
# RVS, and neoplasm codes in data processing
covid_rvs_neoplasm_pattern <- paste(
  c(covid_codes, rvs_codes, neoplasm_codes),
  collapse = "|"
)

claims_query <- paste0("
SELECT
    c.id_series
FROM `pids-drg-data.phic_eclaims.eclaims_", year_to_load, "` c
JOIN `pids-drg-data.phic_hfac.hfac_2023` h
    ON c.id_hci = h.id_hci
WHERE
    c.claim_status = 'G'
    AND c.is_covid = FALSE
    AND c.clin_outpatient = FALSE
    AND h.inst_level IN ('INF', 'L1', 'L2', 'L3');")

claims <- load_or_query(claims_query, "claims", year_to_load)

hci_filter <- hci[inst_level %chin% c("INF", "L1", "L2", "L3"), id_hci]


In [ ]:
library(testthat)
library(data.table)

# Base accepted ICD codes.
acc_pdx_base <- c("I10", "O800", "J189", "A419", "E86", "N179", "Z370")

test_that("Rule 1: CR1 is an acc_pdx", {
  # CR1 valid → source 1.
  claims <- data.table(
    c1 = list(c("I10")), # Valid CR1
    c2 = list(c("X99")), # Invalid
    clin_sdx = list(c("J189")) # Eligible but not used
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 1)
})

test_that("Rule 2: CR2 is an acc_pdx", {
  # CR1 invalid, but CR2 valid → source 2.
  claims <- data.table(
    c1 = list(c("X99")), # Invalid
    c2 = list(c("O800")), # Valid CR2
    clin_sdx = list(c("Z99")) # Invalid
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 2)
})

test_that("Rule 3: Unique best match among SDx similar to CR1", {
  # CR1 (original) is "E869" (not accepted) and clin_sdx has candidates.
  # Similarity: check_similarity("E869", "E86") = 3; check_similarity("E869", "E8691") = 4; "N179" = 0.
  # Unique best is "E8691" → source 3.
  acc_pdx_rule3 <- c("I10", "O800", "J189", "A419", "E86", "E8691", "N179", "Z370")
  claims <- data.table(
    c1 = list(c("E869")), # CR1 original reference ("E869")
    c2 = list(c("X99")), # Invalid
    clin_sdx = list(c("E86", "E8691", "N179"))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_rule3
  )
  expect_equal(result$clin_pdx_source[1], 3)
})

test_that("Rule 4: Tie among SDx similar to CR1, random choice", {
  # CR1 = "E869"; candidates "E86" and "E868" both yield similarity 3.
  # Expect a tie → source 4.
  acc_pdx_rule4 <- c("I10", "O800", "J189", "A419", "E86", "E868", "N179", "Z370")
  claims <- data.table(
    c1 = list(c("E869")), # CR1 reference
    c2 = list(c("X99")), # Invalid
    clin_sdx = list(c("E86", "E868", "N179"))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_rule4
  )
  expect_equal(result$clin_pdx_source[1], 4)
})

test_that("Rule 5: Unique best match among SDx similar to CR2", {
  # CR1 is invalid; CR2 (original) is "E869"; clin_sdx has candidates.
  # Similarity: "E86" = 3; "E8691" = 4; "N179" = 0.
  # Unique best with CR2 is "E8691" → source 5.
  acc_pdx_rule5 <- c("I10", "O800", "J189", "A419", "E86", "E8691", "N179", "Z370")
  claims <- data.table(
    c1 = list(c("X99")), # CR1 invalid
    c2 = list(c("E869")), # CR2 reference
    clin_sdx = list(c("E86", "E8691", "N179"))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_rule5
  )
  expect_equal(result$clin_pdx_source[1], 5)
})

test_that("Rule 6: Tie among SDx similar to CR2, random choice", {
  # CR1 invalid; CR2 = "E869"; candidates "E86" and "E868" both yield similarity 3.
  # Tie → source 6.
  acc_pdx_rule6 <- c("I10", "O800", "J189", "A419", "E86", "E868", "N179", "Z370")
  claims <- data.table(
    c1 = list(c("X99")), # CR1 invalid
    c2 = list(c("E869")), # CR2 reference
    clin_sdx = list(c("E86", "E868", "N179"))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_rule6
  )
  expect_equal(result$clin_pdx_source[1], 6)
})

test_that("Rule 7: Only eligible SDx without any similarity", {
  # CR1 = "X999", CR2 = "Y999" (both not similar), and only one SDx candidate exists.
  # → source 7.
  claims <- data.table(
    c1 = list(c("X999")), # CR1 reference
    c2 = list(c("Y999")), # CR2 reference
    clin_sdx = list(c("E86")) # Only one candidate
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 7)
})

test_that("Rule 8: Multiple eligible SDx with no similarity to CR1 or CR2", {
  # CR1 = "X999", CR2 = "Y999" (no similarity), and multiple SDx candidates exist.
  # → random choice among them → source 8.
  claims <- data.table(
    c1 = list(c("X999")), # CR1 reference (no similarity)
    c2 = list(c("Y999")), # CR2 reference (no similarity)
    clin_sdx = list(c("E86", "N179"))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 8)
})

test_that("Rule 99: No eligible PDx in any field", {
  # All fields invalid/empty → source 99.
  claims <- data.table(
    c1 = list(c("X99")),
    c2 = list(c("Z99")),
    clin_sdx = list(c())
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 99)
})

# Base accepted ICD codes.
acc_pdx_base <- c("I10", "O800", "J189", "A419", "E86", "N179", "Z370")

# Additional Test Cases

# Test 10: CR1 with multiple valid values – rule 1 should always win.
test_that("Extra Test 10: CR1 with multiple valid values", {
  claims <- data.table(
    c1 = list(c("X99", "I10", "O800")), # Only "I10" and "O800" are accepted; first accepted is "I10"
    c2 = list(c("O800")), # Valid but not used since CR1 wins
    clin_sdx = list(c("J189", "A419"))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 1)
})

# Test 11: CR1 original is empty (i.e. no valid original value), but CR2 is provided.
test_that("Extra Test 11: Empty CR1 original, use CR2 similarity", {
  claims <- data.table(
    c1 = list(character(0)), # CR1 original empty
    c2 = list(c("E869")), # CR2 valid (for similarity)
    clin_sdx = list(c("E86", "E8691", "N179"))
  )
  # For CR2, similarity: "E869" vs "E86" = 3; vs "E8691" = 4; best candidate "E8691" → should trigger rule 5.
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 5)
})

# Test 12: CR2 original is empty, but CR1 is provided.
test_that("Extra Test 12: Empty CR2 original, use CR1 similarity", {
  claims <- data.table(
    c1 = list(c("E869")), # CR1 provided
    c2 = list(character(0)), # CR2 empty
    clin_sdx = list(c("E86", "E8691", "N179"))
  )
  # CR1 similarity: "E869" vs "E86" = 3; vs "E8691" = 4 → unique best → rule 3.
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 3)
})

# Test 13: Both CR1 and CR2 original are empty and only one SDx candidate exists.
test_that("Extra Test 13: Both CR1 and CR2 empty with one SDx candidate", {
  claims <- data.table(
    c1 = list(character(0)),
    c2 = list(character(0)),
    clin_sdx = list(c("E86"))
  )
  # No similarity can be computed; only one candidate exists → rule 7.
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 7)
})

# Test 14: Both CR1 and CR2 original are empty with multiple SDx candidates.
test_that("Extra Test 14: Both CR1 and CR2 empty with multiple SDx candidates", {
  claims <- data.table(
    c1 = list(character(0)),
    c2 = list(character(0)),
    clin_sdx = list(c("E86", "N179"))
  )
  # No similarity available; multiple candidates → rule 8.
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 8)
})

# Test 15: CR1 contains NA values along with a valid code.
test_that("Extra Test 15: CR1 with NA values", {
  claims <- data.table(
    c1 = list(c(NA, "I10")), # After cleaning, "I10" remains → rule 1.
    c2 = list(c("X99")),
    clin_sdx = list(c("J189"))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 1)
})

# Test 16: All values in clin_sdx are NA → should return rule 99.
test_that("Extra Test 16: Clin_sdx with all NA values", {
  claims <- data.table(
    c1 = list(c("X99")),
    c2 = list(c("Z99")),
    clin_sdx = list(c(NA, NA))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 99)
})

# Test 17: Duplicate SDx codes – should treat duplicates as ties.
test_that("Extra Test 17: Duplicate SDx codes", {
  claims <- data.table(
    c1 = list(c("E869")), # CR1 reference: "E869"
    c2 = list(c("X99")),
    clin_sdx = list(c("E86", "E86", "N179")) # Two duplicates of "E86"
  )
  # "E869" vs "E86" gives similarity of 3; duplicate "E86" yields a tie → rule 4.
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 4)
})

# Test 18: Duplicate best candidate yields tie (best candidate appears more than once).
test_that("Extra Test 18: Duplicate best candidate yields tie", {
  acc_pdx_rule_dup <- c("I10", "O800", "J189", "A419", "E86", "E8691", "N179", "Z370")
  claims <- data.table(
    c1 = list(c("E869")), # CR1: "E869"
    c2 = list(c("X99")),
    clin_sdx = list(c("E86", "E8691", "E8691", "N179")) # "E8691" appears twice → tie → rule 4.
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_rule_dup
  )
  expect_equal(result$clin_pdx_source[1], 4)
})

# Test 19: Both CR1 and CR2 are valid – rule 1 should override.
test_that("Extra Test 19: Both CR1 and CR2 valid, rule 1 overrides", {
  claims <- data.table(
    c1 = list(c("I10", "X99")), # CR1 valid
    c2 = list(c("E869")), # CR2 valid but not used
    clin_sdx = list(c("E86", "N179"))
  )
  result <- find_pdx(
    c1_arg = claims$c1,
    c2_arg = claims$c2,
    clin_sdx_arg = claims$clin_sdx,
    seed = 123,
    accpdx = acc_pdx_base
  )
  expect_equal(result$clin_pdx_source[1], 1)
})
